# general.ipynb — Clean & Normalize Epiconsult General Services CSV

This notebook:
- Reads `general.csv` from your **project root**
- Detects each **service category** from the `"Listing of ..."` lines
- Builds one clean table with: **Category, S/N, Name, and dynamic price columns**
- Converts prices to **₦ formatted strings**
- Converts blanks (and optionally zeros) to **N/A**
- Overwrites `general.csv` safely (temp write → replace)

> You can change settings in Cell 1.


In [5]:
# --- Cell 1: Setup ---
from pathlib import Path
import csv
import re
import pandas as pd
import tempfile
import os

INPUT_CSV = Path("general.csv")          # in your project root
OUTPUT_CSV = Path("general_new.csv")         # overwrite (keeps only one general CSV)

# If a price cell is blank -> N/A (recommended for frontend display)
BLANK_TO_NA = True

# In your file, 0 typically means "not available" for that column (e.g., Outsourced = 0)
# Set to True to convert explicit zeros to N/A too.
ZERO_TO_NA = True

# Always format money as ₦#,###.## when available
FORMAT_NAIRA = True


In [6]:
# --- Cell 2: Read raw CSV rows (preserve structure) ---
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Not found: {INPUT_CSV.resolve()}")

# Use utf-8-sig so files saved with BOM still read cleanly
raw_text = INPUT_CSV.read_text(encoding="utf-8-sig", errors="replace").splitlines()

rows = []
reader = csv.reader(raw_text)
for r in reader:
    rows.append([c.strip() for c in r])

print("Total raw rows:", len(rows))
print("Sample rows (first 10):")
for i in range(min(10, len(rows))):
    print(rows[i])


Total raw rows: 87
Sample rows (first 10):
['Epiconsult Diagnostics - Service Charges Listing as at 03-Jan-2026', '', '', '', '']
['General Service Charges Listing as at 03-Jan-2026', '', '', '', '']
['', '', '', '', '']
['Listing of Clinical Consultation', '', '', '', '']
['', '', '', '', '']
['S/N', 'Name', 'Outsourced (B)', 'Walk in Patient (C)', 'Hospital Patient (D)']
['1', 'GOPD', '5,000.00', '5,000.00', '1,000.00']
['2', 'CONSULTATION WITH PAEDIATRICIAN', '15,000.00', '15,000.00', '4,000.00']
['3', 'GENERAL MEDICAL CONSULTATION/7 DAYS', '10,000.00', '10,000.00', '3,000.00']
['4', 'Dental Consultation', '8,000.00', '8,000.00', '8,000.00']


In [7]:
# --- Cell 3: Parse categories + tables into one normalized table ---

def clean_category(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"^Listing\s+of\s+", "", s, flags=re.IGNORECASE).strip()

    # Small normalization fixes
    s = s.replace("MedicalREports", "Medical Reports")
    s = s.replace("MedicalREport", "Medical Report")
    s = s.replace("REGISTRATION", "Registration")
    s = s.replace("Clinical Consultation", "Clinical Consultation")  # keep as-is

    # Title-case but keep common abbreviations
    # (only for category labels, not service names)
    words = []
    for w in s.split():
        if w.upper() in {"GOPD", "RVS", "COVID"}:
            words.append(w.upper())
        else:
            words.append(w.capitalize())
    return " ".join(words).strip() or "Uncategorized"

def normalize_header(cols):
    # Keep column names dynamic but remove extra whitespace
    out = []
    for c in cols:
        c = (c or "").strip()
        c = re.sub(r"\s+", " ", c)
        out.append(c)
    return out

def parse_money(cell: str):
    """Return numeric float if parseable, else None."""
    if cell is None:
        return None
    s = str(cell).strip().strip('"').strip("'")
    if s == "":
        return None

    # Remove currency and commas
    s2 = s.replace("₦", "").replace(",", "").strip()

    # Some cells might contain text; accept only number-like
    if not re.fullmatch(r"-?\d+(?:\.\d+)?", s2):
        return None

    try:
        return float(s2)
    except:
        return None

def format_naira(v: float) -> str:
    return f"₦{v:,.2f}"

current_category = "Uncategorized"
current_header = None   # full header row: ["S/N","Name",...]
records = []

for r in rows:
    # Skip completely empty rows
    if not any((c or "").strip() for c in r):
        continue

    first = (r[0] or "").strip()

    # Category marker
    if re.search(r"\bListing\s+of\b", first, flags=re.IGNORECASE):
        current_category = clean_category(first)
        current_header = None
        continue

    # Header row marker (table start)
    if len(r) >= 2 and first.upper() == "S/N" and (r[1] or "").strip().lower() == "name":
        current_header = normalize_header(r)
        continue

    # Data row (requires a header and a numeric S/N)
    if current_header and first.isdigit():
        sn = int(first)
        name = (r[1] if len(r) > 1 else "").strip()
        if not name:
            continue

        price_cols = current_header[2:]  # dynamic columns
        prices = {}

        for idx, col_name in enumerate(price_cols, start=2):
            cell = r[idx] if idx < len(r) else ""
            num = parse_money(cell)

            # Decide N/A rules
            if num is None:
                val = "N/A" if BLANK_TO_NA else ""
            else:
                if ZERO_TO_NA and abs(num) < 1e-12:
                    val = "N/A"
                else:
                    val = format_naira(num) if FORMAT_NAIRA else str(num)

            prices[col_name] = val

        rec = {
            "Category": current_category,
            "S/N": sn,
            "Name": name,
            **prices
        }
        records.append(rec)

df = pd.DataFrame(records)

print("Parsed rows:", len(df))
print("Categories:", df["Category"].nunique() if len(df) else 0)
df.head(10)


Parsed rows: 53
Categories: 6


,Category,S/N,Name,Outsourced (B),Walk in Patient (C),Hospital Patient (D)
0,Clinical Consultation,1,GOPD,"₦5,000.00","₦5,000.00","₦1,000.00"
1,Clinical Consultation,2,CONSULTATION WITH PAEDIATRICIAN,"₦15,000.00","₦15,000.00","₦4,000.00"
2,Clinical Consultation,3,GENERAL MEDICAL CONSULTATION/7 DAYS,"₦10,000.00","₦10,000.00","₦3,000.00"
3,Clinical Consultation,4,Dental Consultation,"₦8,000.00","₦8,000.00","₦8,000.00"
4,Clinical Consultation,5,COSMETOLOGY,"₦5,000.00","₦5,000.00","₦5,000.00"
5,Clinical Consultation,6,OBSERVATION,"₦10,000.00","₦10,000.00","₦5,000.00"
6,Clinical Consultation,7,IN-HOUSE SPECIALIST - CONSULTATION/7 DAYS,"₦15,000.00","₦15,000.00","₦15,000.00"
7,Clinical Consultation,8,VISITING CONSULTANT - CONSULTATION,"₦25,000.00","₦20,000.00","₦15,000.00"
8,Clinical Consultation,9,PHYSIOTHERAPY,"₦15,000.00","₦15,000.00","₦15,000.00"
9,Clinical Consultation,10,GENERAL CONSULTATION,"₦3,000.00","₦3,000.00","₦3,000.00"


In [8]:
# --- Cell 4: Sort + write cleaned CSV (safe overwrite) ---
if df.empty:
    raise ValueError("No records parsed. Check the input file format.")

df = df.sort_values(["Category", "S/N"], kind="stable").reset_index(drop=True)

summary = df.groupby("Category")["Name"].count().sort_values(ascending=False)
print("Rows per category:")
print(summary)

# IMPORTANT: write UTF-8 with BOM so ₦ displays correctly in Excel/Windows
with tempfile.NamedTemporaryFile("w", delete=False, suffix=".csv", newline="") as tmp:
    tmp_path = Path(tmp.name)
    df.to_csv(tmp_path, index=False, lineterminator="\n", encoding="utf-8-sig")

os.replace(tmp_path, OUTPUT_CSV)
print(f"Saved cleaned CSV to: {OUTPUT_CSV.resolve()}")

df.head(20)


Rows per category:
Category
Procedures               21
Clinical Consultation    15
Dental Services          12
Consumables               3
Medical Reports           1
Registration              1
Name: Name, dtype: int64
Saved cleaned CSV to: C:\Users\ACEMX\Desktop\LLMprojects\e-CLINIC\general_new.csv


,Category,S/N,Name,Outsourced (B),Walk in Patient (C),Hospital Patient (D)
0,Clinical Consultation,1,GOPD,"₦5,000.00","₦5,000.00","₦1,000.00"
1,Clinical Consultation,2,CONSULTATION WITH PAEDIATRICIAN,"₦15,000.00","₦15,000.00","₦4,000.00"
2,Clinical Consultation,3,GENERAL MEDICAL CONSULTATION/7 DAYS,"₦10,000.00","₦10,000.00","₦3,000.00"
3,Clinical Consultation,4,Dental Consultation,"₦8,000.00","₦8,000.00","₦8,000.00"
4,Clinical Consultation,5,COSMETOLOGY,"₦5,000.00","₦5,000.00","₦5,000.00"
5,Clinical Consultation,6,OBSERVATION,"₦10,000.00","₦10,000.00","₦5,000.00"
6,Clinical Consultation,7,IN-HOUSE SPECIALIST - CONSULTATION/7 DAYS,"₦15,000.00","₦15,000.00","₦15,000.00"
7,Clinical Consultation,8,VISITING CONSULTANT - CONSULTATION,"₦25,000.00","₦20,000.00","₦15,000.00"
8,Clinical Consultation,9,PHYSIOTHERAPY,"₦15,000.00","₦15,000.00","₦15,000.00"
9,Clinical Consultation,10,GENERAL CONSULTATION,"₦3,000.00","₦3,000.00","₦3,000.00"
